# 15.06 · Reading papers & staying current

**Chapter:** Advanced & Frontier Topics · **Milestone:** M7

## Goals
- [ ] Read a paper in **three passes** and judge its evidence (baselines, ablations, evals, compute, contamination)
- [ ] Build a personal **paper radar**: HF Daily Papers + arXiv, ranked by *your* interests, with LLM-written first-pass summaries
- [ ] Follow the field without drowning: a small set of feeds, leaderboards and a weekly routine
- [ ] Turn hype into a decision: test claims on **your** eval set

## Resources
- [S. Keshav – How to Read a Paper](https://web.stanford.edu/class/ee384m/Handouts/HowtoReadPaper.pdf) — free
- [Hugging Face Papers](https://huggingface.co/papers) (daily curated papers, with discussion) · [arXiv listing cs.CL](https://arxiv.org/list/cs.CL/recent) — free
- Blogs / newsletters: [Simon Willison](https://simonwillison.net/), [Latent Space](https://www.latent.space/), [Interconnects](https://www.interconnects.ai/),
  [AI News (smol.ai)](https://news.smol.ai/), [Ahead of AI (Raschka)](https://magazine.sebastianraschka.com/), [Lilian Weng](https://lilianweng.github.io/) — free
- Leaderboards: [LMArena](https://lmarena.ai/), [Artificial Analysis](https://artificialanalysis.ai/), [OpenRouter rankings](https://openrouter.ai/rankings),
  [SWE-bench](https://www.swebench.com/), [Open ASR](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard), [MTEB](https://huggingface.co/spaces/mteb/leaderboard) — free

## How to read a paper: three passes
1. **Five minutes** — title, abstract, figures, conclusion, section headings. Answer: what problem, what claim, is it relevant to me? Most papers stop here.
2. **One hour** — read the body, skip proofs. Look hard at **evidence**: what baselines (strong and recent, or strawmen?), which benchmarks
   (saturated? possibly in the training data?), ablations (which part actually matters?), variance (one seed?), compute (fair comparison?).
3. **A few hours** — re-derive the key equations, read the code, try to **reproduce** one figure or number. This is where you really learn it.

Red flags: only self-chosen benchmarks, no ablation, comparisons at different compute or data, "SOTA" by 0.3 points without error bars,
no code or weights for an empirical claim. Green flags: simple idea + strong baselines + ablations + released code.

You already did third passes in this course: the s1 budget-forcing curve (15.01), induction heads (15.04), DPO's failure mode (12.04).

In [ ]:
import json
import re
import time
import xml.etree.ElementTree as ET
from datetime import date, timedelta

import httpx
import numpy as np
from pydantic import BaseModel

from aieng import DATA, rag
from aieng.llm import client

## 1. Collect: HF Daily Papers + arXiv
HF Daily Papers is a human-curated shortlist (upvotes, discussion, often links to code/models). arXiv is the firehose: ~1,000 new
cs.AI/cs.CL/cs.LG papers per weekday. Both have free APIs.

In [ ]:
def hf_daily(days=3):
    papers = []
    for d in range(days):
        day = (date.today() - timedelta(days=d)).isoformat()
        r = httpx.get("https://huggingface.co/api/daily_papers", params={"date": day}, timeout=30)
        if r.status_code == 200:
            for p in r.json():
                pp = p["paper"]
                papers.append({"id": pp["id"], "title": pp["title"].strip(), "abstract": pp.get("summary", "").strip(),
                               "upvotes": pp.get("upvotes", 0), "source": "hf", "url": f"https://huggingface.co/papers/{pp['id']}"})
    return papers


def arxiv_recent(query="cat:cs.CL OR cat:cs.LG OR cat:cs.AI", n=100):
    r = httpx.get("http://export.arxiv.org/api/query", timeout=60, follow_redirects=True,
                  params={"search_query": query, "sortBy": "submittedDate", "sortOrder": "descending", "max_results": n})
    ns = {"a": "http://www.w3.org/2005/Atom"}
    out = []
    for e in ET.fromstring(r.text).findall("a:entry", ns):
        aid = e.find("a:id", ns).text.rsplit("/", 1)[-1]
        out.append({"id": aid, "title": " ".join(e.find("a:title", ns).text.split()), "abstract": " ".join(e.find("a:summary", ns).text.split()),
                    "upvotes": 0, "source": "arxiv", "url": f"https://arxiv.org/abs/{aid}"})
    return out


papers = {p["id"]: p for p in hf_daily() + arxiv_recent()}.values()  # dedupe by id (HF wins: it has upvotes)
papers = list(papers)
print(f"{len(papers)} papers ({sum(p['source'] == 'hf' for p in papers)} from HF Daily, rest from arXiv)")
for p in papers[:3]:
    print(f"- [{p['source']}] {p['title'][:90]}")

## 2. Rank by your interests
Write your interests as short descriptions, embed them and every abstract (07.x), and score each paper by its best-matching interest.
Add a small bonus for community signal (HF upvotes). Edit `INTERESTS` to your own — this is the part that makes the radar *yours*.

In [ ]:
INTERESTS = [
    "retrieval-augmented generation, rerankers and embeddings for search",
    "evaluating LLM applications, LLM-as-a-judge, benchmarks and their contamination",
    "small language models, distillation, fine-tuning with LoRA on consumer hardware",
    "LLM agents with tools, long-horizon tasks, coding agents",
    "efficient inference: quantization, speculative decoding, KV cache",
]
P = rag.embed([p["title"] + ". " + p["abstract"][:1000] for p in papers])
I = rag.embed(INTERESTS, query=True)
sims = P @ I.T
for p, s in zip(papers, sims):
    p["interest"] = INTERESTS[int(s.argmax())].split(",")[0]
    p["score"] = float(s.max()) + 0.01 * np.log1p(p["upvotes"])
ranked = sorted(papers, key=lambda p: -p["score"])
for p in ranked[:10]:
    print(f"{p['score']:.3f}  [{p['interest'][:28]:28}] {p['title'][:80]}")

## 3. First pass, automated: structured summaries
An LLM does a decent **first pass** (what problem, what claim, what evidence) for the top papers — it does *not* replace your second pass.
Structured output (05.04) makes the digest scannable and lets you filter ("only papers with code", "only strong evidence").

In [ ]:
class FirstPass(BaseModel):
    problem: str
    method: str
    key_result: str
    evidence: str  # what baselines/benchmarks back the claim
    evidence_strength: str  # weak | moderate | strong
    worth_second_pass: bool
    why: str


c, MODEL = client()
SUMMARY_PROMPT = f"""You are helping an AI engineer triage papers. Their interests: {INTERESTS}.
From the title and abstract only, fill this JSON (each text field ≤ 25 words):
{{"problem": ..., "method": ..., "key_result": ..., "evidence": ..., "evidence_strength": "weak|moderate|strong",
 "worth_second_pass": true|false, "why": ...}}
Be skeptical: abstracts oversell. "strong" requires named strong baselines or standard benchmarks with clear margins."""


def first_pass(p):
    r = c.chat.completions.create(model=MODEL, temperature=0, max_tokens=600, messages=[
        {"role": "system", "content": SUMMARY_PROMPT}, {"role": "user", "content": f"Title: {p['title']}\n\nAbstract: {p['abstract']}"}])
    text = r.choices[0].message.content
    return FirstPass.model_validate_json(text[text.index("{"): text.rindex("}") + 1])


digest = []
for p in ranked[:6]:
    try:
        fp = first_pass(p)
    except Exception as e:
        print("skip:", p["title"][:60], type(e).__name__); continue
    digest.append((p, fp))
    print(f"\n{'⭐' if fp.worth_second_pass else '  '} {p['title']}\n   {p['url']}\n   problem : {fp.problem}\n   method  : {fp.method}\n"
          f"   result  : {fp.key_result}\n   evidence: {fp.evidence} ({fp.evidence_strength})\n   why     : {fp.why}")
    time.sleep(2)

Count the ⭐s. LLM triage is **generous**: the top-ranked papers match your interests by construction, and abstracts are written to sell,
so the model tends to recommend most of them and to call evidence "strong" from the abstract alone. Treat its output as a reading queue,
not a verdict — and calibrate it: mark your own keep/skip for a week and compare (exercise 2).

Save the digest as Markdown (or send it to Slack/email weekly with the scheduling tools you use). Keep what you read in one place:
a running notes file with "paper → one-line takeaway → did I try it?" is worth more than 500 bookmarks.

In [ ]:
md = [f"# Paper radar — {date.today().isoformat()}\n"]
for p, fp in digest:
    md.append(f"## {'⭐ ' if fp.worth_second_pass else ''}[{p['title']}]({p['url']})\n- **Problem**: {fp.problem}\n- **Method**: {fp.method}\n"
              f"- **Result**: {fp.key_result}\n- **Evidence** ({fp.evidence_strength}): {fp.evidence}\n- **Why**: {fp.why}\n")
out = DATA / f"paper-radar-{date.today().isoformat()}.md"
out.write_text("\n".join(md))
print(f"wrote {out} ({len(digest)} papers)")

## 4. Blogs and newsletters: a small, high-signal feed list
Practitioner blogs often explain what matters *for builders* weeks before it's in a paper (or instead of one). RSS/Atom is plain XML:

In [ ]:
FEEDS = {
    "Simon Willison": "https://simonwillison.net/atom/everything/",
    "Latent Space": "https://www.latent.space/feed",
    "Interconnects": "https://www.interconnects.ai/feed",
    "AI News (smol.ai)": "https://news.smol.ai/rss.xml",
}


def latest(url, n=3):
    root = ET.fromstring(httpx.get(url, timeout=30, follow_redirects=True).content)
    items = root.findall(".//item") or root.findall("{http://www.w3.org/2005/Atom}entry")
    out = []
    for it in items[:n]:
        title = it.findtext("title") or it.findtext("{http://www.w3.org/2005/Atom}title") or ""
        when = it.findtext("pubDate") or it.findtext("{http://www.w3.org/2005/Atom}updated") or ""
        out.append((" ".join(title.split())[:90], when[:16]))
    return out


for name, url in FEEDS.items():
    try:
        print(f"\n{name}:"); [print(f"   {when:16}  {t}") for t, when in latest(url)]
    except Exception as e:
        print(f"   ({type(e).__name__})")

## 5. From hype to decision
New model or technique announced? The engineer's loop:
1. **Read the claim precisely** (which benchmark, which baseline, which settings). Check independent evals (Artificial Analysis, LMArena, community replications).
2. **Check it applies to you**: your languages, your context lengths, your latency and cost budget, your license constraints (15.03).
3. **Run your eval set** (chapter 10): with `aieng.llm`, swapping a model is one string. A model that's #1 on a leaderboard can be worse on your task.
4. **Decide and write it down** (date, numbers, decision) so you don't re-litigate it next month.

**A sustainable routine** (~2 h/week): skim the radar on Monday (pass 1 on ~10 papers, pass 2 on 1), read 2–3 blog posts, try one new thing
hands-on on Friday (a model on your eval, a library on a toy problem). Depth beats breadth: one reproduced result teaches more than 50 abstracts.

## Exercises
1. Reproduce one figure from a recent paper: pick a paper from your radar with code and a small-scale experiment (a plot that runs on a
   laptop or a free Colab GPU). Write up what matched and what didn't.
2. Edit `INTERESTS`, run the radar for a week (once a day) and keep only papers you'd actually read. Tune the ranking: does adding upvotes,
   recency, or "has code" (`githubRepo` in the HF API) improve it?
3. Do a full **three-pass read** of one classic: "Attention Is All You Need" (in `data/attention.pdf`), the RAG paper, or DPO. Write a one-page
   summary: claim, evidence, what you'd test next.
4. Pick a model released in the last month and run your M3/M4 eval suite on it against your current model. Make a keep/switch decision with numbers.

In [ ]:
# your code here